# Inferring hidden regimes from filtered probabilities

## Respecting the training boundary in public returns

For an index level \(P_t>0\), the daily log return in percentage points is

$$
r_t=100\left(\log P_t-\log P_{t-1}\right).
$$

Here, \(P_t\) is the public NASDAQ Composite value on date \(t\), \(P_{t-1}\) is the preceding valid value, and \(r_t\) is the continuously compounded daily return measured in percentage points. Removing unavailable levels before taking differences prevents a missing observation from being treated as a zero return.

In [1]:
from pathlib import Path
import hashlib
import sys
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)
from scipy.stats import norm
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset

import RiskLabAI
from RiskLabAI.backtest.adaptation import prequential_adaptation
from RiskLabAI.data.labeling import cusum_filter_events_dynamic_threshold, symmetric_cusum_filter
from RiskLabAI.features.structural_breaks.pelt_change_points import pelt_change_points
from river.drift import ADWIN

assert RiskLabAI.__version__ == "3.2.0"

warnings.filterwarnings("ignore", message="New behaviour in v1.1.5")

DATA_PATH = ensure_dataset(BOOK_ROOT, "fred_nasdaq")
FIGURES = BOOK_ROOT / "outputs" / "figures" / "13_regime_adaptation"
TABLES = BOOK_ROOT / "outputs" / "tables" / "13_regime_adaptation"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "13_regime_adaptation"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return "0"
        if number.is_integer():
            return f"{int(number):,}"
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f"{number:.6g}"
        return f"{number:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)


def load_public_data():
    expected_hash = "2E988948FFCB108DA4902DFEDAA5DF756E7E102DC523A99A8916A0533291F2D7"
    assert DATA_PATH.is_file()
    assert sha256(DATA_PATH) == expected_hash
    raw = pd.read_csv(DATA_PATH)
    raw["Date"] = pd.to_datetime(raw["observation_date"])
    raw["NASDAQ Composite"] = pd.to_numeric(raw["NASDAQCOM"], errors="coerce")
    data = raw.dropna(subset=["NASDAQ Composite"]).sort_values("Date").reset_index(drop=True)
    data["Daily log return (%)"] = 100 * np.log(data["NASDAQ Composite"] / data["NASDAQ Composite"].shift(1))
    data["Log index level"] = 100 * np.log(data["NASDAQ Composite"])
    return data


def prepare_sequential_data(data, min_samples=756):
    frame = data[["Date", "Daily log return (%)"]].copy()
    lagged = frame["Daily log return (%)"].shift(1)
    frame["Previous return (%)"] = lagged
    frame["Previous absolute return (%)"] = lagged.abs()
    frame["Trailing 5 day mean (%)"] = lagged.rolling(5).mean()
    frame["Trailing 21 day volatility (%)"] = lagged.rolling(21).std(ddof=0)
    frame["Trailing 63 day volatility (%)"] = lagged.rolling(63).std(ddof=0)
    frame = frame.dropna().reset_index(drop=True)
    threshold = float(frame.loc[: min_samples - 1, "Daily log return (%)"].abs().quantile(0.80))
    frame["Large move"] = (frame["Daily log return (%)"].abs() > threshold).astype(int)
    return frame, threshold


def feature_columns():
    return [
        "Previous return (%)",
        "Previous absolute return (%)",
        "Trailing 5 day mean (%)",
        "Trailing 21 day volatility (%)",
        "Trailing 63 day volatility (%)",
    ]


def make_estimator():
    return Pipeline([
        ("standardization", StandardScaler()),
        ("logistic regression", LogisticRegression(C=1, class_weight="balanced", random_state=314159, max_iter=1000)),
    ])


def zero_one_loss(actual, prediction):
    return float(actual != prediction)


def run_adaptive(features, targets, delta=0.01, min_samples=756, window_size=756):
    return prequential_adaptation(
        features,
        targets,
        make_estimator(),
        loss_function=zero_one_loss,
        window_size=window_size,
        min_samples=min_samples,
        delta=delta,
    )


def run_static(features, targets, min_samples=756):
    predictions = np.full(len(targets), np.nan)
    losses = np.full(len(targets), np.nan)
    model = clone(make_estimator()).fit(features[:min_samples], targets[:min_samples])
    predictions[min_samples:] = model.predict(features[min_samples:])
    losses[min_samples:] = (predictions[min_samples:] != targets[min_samples:]).astype(float)
    return {"predictions": predictions, "losses": losses, "refit_indices": np.array([min_samples - 1], dtype=int)}


def run_periodic(features, targets, min_samples=756, window_size=756, interval=252):
    predictions = np.full(len(targets), np.nan)
    losses = np.full(len(targets), np.nan)
    refits = [min_samples - 1]
    model = clone(make_estimator()).fit(features[:min_samples], targets[:min_samples])
    for index in range(min_samples, len(targets)):
        predictions[index] = model.predict(features[index : index + 1])[0]
        losses[index] = float(predictions[index] != targets[index])
        if (index + 1 - min_samples) % interval == 0:
            begin = max(0, index + 1 - window_size)
            model = clone(make_estimator()).fit(features[begin : index + 1], targets[begin : index + 1])
            refits.append(index)
    return {"predictions": predictions, "losses": losses, "refit_indices": np.array(refits, dtype=int)}

In [2]:
data = load_public_data()

provenance = pd.DataFrame([
    {"Field": "Source", "Value": "Federal Reserve Economic Data"},
    {"Field": "Series", "Value": "NASDAQ Composite, NASDAQCOM"},
    {"Field": "Observation period", "Value": f"{data['Date'].min().date()} to {data['Date'].max().date()}"},
    {"Field": "Public URL", "Value": "https://fred.stlouisfed.org/series/NASDAQCOM"},
    {"Field": "Local file SHA256", "Value": sha256(DATA_PATH)},
    {"Field": "Download during execution", "Value": "No"},
])
save_table(provenance, "data_provenance", "Public data provenance for the regime and adaptation calculations.", "tab:chapter14-data-provenance", fit=True)

summary = pd.DataFrame([
    {"Quantity": "Valid index levels", "Value": len(data)},
    {"Quantity": "Daily log returns", "Value": data["Daily log return (%)"].notna().sum()},
    {"Quantity": "First valid date", "Value": data["Date"].min()},
    {"Quantity": "Last valid date", "Value": data["Date"].max()},
    {"Quantity": "Training cutoff", "Value": pd.Timestamp("2014-12-31")},
    {"Quantity": "Post-cutoff return observations", "Value": (data["Date"] > pd.Timestamp("2014-12-31")).sum()},
])
save_table(summary, "public_data_summary", "Public NASDAQ Composite sample used in Chapter 14.", "tab:chapter14-public-data-summary")

history = data[["Date", "NASDAQ Composite", "Daily log return (%)"]].copy()
history.to_csv(FIGURE_DATA / "public_price_return_history.csv", index=False)
figure, axes = plt.subplots(2, 1, figsize=(11.2, 7.0), sharex=True, constrained_layout=True)
axes[0].plot(history["Date"], history["NASDAQ Composite"], color=PALETTE["cyan"], linewidth=1.25)
axes[0].set_yscale("log")
axes[0].set_ylabel("Index level")
axes[0].set_title("Public NASDAQ Composite Prices and Daily Returns")
axes[1].plot(history["Date"], history["Daily log return (%)"], color=PALETTE["amber"], linewidth=0.58)
axes[1].set_ylabel("Daily log return (%)")
axes[1].set_xlabel("Date")
for axis in axes:
    finish_axis(axis)
save_figure(figure, "public_price_return_history")
plt.close(figure)

,Field,Value
0,Source,Federal Reserve Economic Data
1,Series,"NASDAQ Composite, NASDAQCOM"
2,Observation period,1994-01-03 to 2025-12-31
3,Public URL,https://fred.stlouisfed.org/series/NASDAQCOM
4,Local file SHA256,2E988948FFCB108DA4902DFEDAA5DF756E7E102DC523A9...
5,Download during execution,No


,Quantity,Value
0,Valid index levels,"8,055"
1,Daily log returns,"8,054"
2,First valid date,1994-01-03
3,Last valid date,2025-12-31
4,Training cutoff,2014-12-31
5,Post-cutoff return observations,"2,767"


## A Two State Gaussian Return Model

Let the hidden state on date \(t\) be \(S_t\in\{0,1\}\). Conditional on that state, the return is modeled as

$$
r_t=\mu_{S_t}+\sigma_{S_t}\varepsilon_t,
\qquad
\varepsilon_t\sim N(0,1).
$$

Here, \(\mu_j\) is the mean return in state \(j\), \(\sigma_j>0\) is its standard deviation, and \(\varepsilon_t\) is a standard normal innovation. The state follows a Markov chain with

$$
p_{ij}=\Pr(S_t=j\mid S_{t-1}=i),
\qquad
\sum_{j=0}^{1}p_{ij}=1.
$$

The parameter \(p_{ij}\) is the probability of moving from state \(i\) yesterday to state \(j\) today. The transition probabilities are constant in this model. They describe persistence in the fitted statistical states, not permanent economic regimes.

In [3]:
returns = data.loc[data["Daily log return (%)"].notna(), ["Date", "Daily log return (%)"]].reset_index(drop=True)
training_cutoff = pd.Timestamp("2014-12-31")
training = returns.loc[returns["Date"] <= training_cutoff].copy()
training_model = MarkovRegression(training["Daily log return (%)"].to_numpy(), k_regimes=2, trend="c", switching_variance=True)
fit_result = training_model.fit(disp=False, maxiter=1000)
assert bool(fit_result.mle_retvals.get("converged", False))

parameters = np.asarray(fit_result.params, dtype=float)
parameter_map = dict(zip(training_model.param_names, parameters))
variances = np.array([parameter_map["sigma2[0]"], parameter_map["sigma2[1]"]])
state_order = np.argsort(variances)
low_state, high_state = map(int, state_order)
state_names = {low_state: "Lower volatility state", high_state: "Higher volatility state"}

transition_raw = np.asarray(training_model.regime_transition_matrix(parameters))[:, :, 0].T
transition = transition_raw[np.ix_(state_order, state_order)]

sample_table = pd.DataFrame([
    {"Quantity": "Fitting start", "Value": training["Date"].min()},
    {"Quantity": "Fitting end", "Value": training["Date"].max()},
    {"Quantity": "Fitting observations", "Value": len(training)},
    {"Quantity": "Post-cutoff observations", "Value": len(returns) - len(training)},
    {"Quantity": "Log likelihood", "Value": float(fit_result.llf)},
    {"Quantity": "Optimizer converged", "Value": bool(fit_result.mle_retvals.get("converged", False))},
])
save_table(sample_table, "state_model_sample", "Fitting sample and numerical status of the two state model.", "tab:chapter14-state-model-sample")

state_parameters = pd.DataFrame([
    {
        "State label": state_names[state],
        "Original numerical state": state,
        "Mean return (%)": parameter_map[f"const[{state}]"],
        "Variance": parameter_map[f"sigma2[{state}]"],
        "Standard deviation (%)": np.sqrt(parameter_map[f"sigma2[{state}]"]),
    }
    for state in state_order
])
save_table(state_parameters, "state_parameters", "Fitted Gaussian parameters ordered by return variance.", "tab:chapter14-state-parameters")

transition_table = pd.DataFrame([
    {"State today": state_names[state_order[i]], "State tomorrow": state_names[state_order[j]], "Probability": transition[i, j]}
    for i in range(2)
    for j in range(2)
])
save_table(transition_table, "transition_probabilities", "Estimated transition probabilities between the two fitted states.", "tab:chapter14-transition-probabilities")

durations = pd.DataFrame([
    {"State": state_names[state_order[index]], "Stay probability": transition[index, index], "Expected duration in trading days": 1 / (1 - transition[index, index])}
    for index in range(2)
])
save_table(durations, "expected_state_durations", "Expected state durations implied by constant transition probabilities.", "tab:chapter14-expected-state-durations")

grid = np.linspace(returns["Daily log return (%)"].quantile(0.001), returns["Daily log return (%)"].quantile(0.999), 600)
density_rows = []
figure, axis = plt.subplots(figsize=(9.8, 5.6), constrained_layout=True)
for state, color in [(low_state, PALETTE["cyan"]), (high_state, PALETTE["rose"])]:
    mean = parameter_map[f"const[{state}]"]
    standard_deviation = np.sqrt(parameter_map[f"sigma2[{state}]"])
    density = norm.pdf(grid, loc=mean, scale=standard_deviation)
    axis.plot(grid, density, color=color, linewidth=2.1, label=state_names[state])
    density_rows.extend({"Daily log return (%)": value, "Density": height, "State": state_names[state]} for value, height in zip(grid, density))
axis.set_xlabel("Daily log return (%)")
axis.set_ylabel("Conditional density")
axis.set_title("Fitted Return Distribution in Each State")
axis.legend()
finish_axis(axis)
pd.DataFrame(density_rows).to_csv(FIGURE_DATA / "state_density_curves.csv", index=False)
save_figure(figure, "state_density_distributions")
plt.close(figure)

transition_figure_data = transition_table.copy()
transition_figure_data.to_csv(FIGURE_DATA / "transition_matrix.csv", index=False)
figure, axis = plt.subplots(figsize=(7.8, 5.8), constrained_layout=True)
image = axis.imshow(transition, vmin=0, vmax=1, cmap="magma")
short_labels = ["Lower volatility", "Higher volatility"]
axis.set_xticks([0, 1], short_labels)
axis.set_yticks([0, 1], short_labels)
axis.set_xlabel("State tomorrow")
axis.set_ylabel("State today")
axis.set_title("Estimated State Transition Probabilities")
for i in range(2):
    for j in range(2):
        axis.text(j, i, f"{transition[i, j]:.4f}", ha="center", va="center", color="white" if transition[i, j] < 0.6 else PALETTE["background"], fontsize=12)
colorbar = figure.colorbar(image, ax=axis)
colorbar.set_label("Probability")
save_figure(figure, "transition_matrix")
plt.close(figure)

,Quantity,Value
0,Fitting start,1994-01-04
1,Fitting end,2014-12-31
2,Fitting observations,"5,287"
3,Post-cutoff observations,"2,767"
4,Log likelihood,-8953.767068
5,Optimizer converged,Pass


,State label,Original numerical state,Mean return (%),Variance,Standard deviation (%)
0,Lower volatility state,0,0.095185,0.833288,0.912846
1,Higher volatility state,1,-0.072152,5.560235,2.358015


,State today,State tomorrow,Probability
0,Lower volatility state,Lower volatility state,0.994955
1,Lower volatility state,Higher volatility state,0.005045
2,Higher volatility state,Lower volatility state,0.009189
3,Higher volatility state,Higher volatility state,0.990811


,State,Stay probability,Expected duration in trading days
0,Lower volatility state,0.994955,198.214583
1,Higher volatility state,0.990811,108.830466


## Filtering with Frozen Parameters

Suppose \(\pi_{t-1\mid t-1,i}\) is the filtered probability of state \(i\) after observing returns through date \(t-1\). The one-step state prediction is

$$
\pi_{t\mid t-1,j}
=\sum_{i=0}^{1}p_{ij}\pi_{t-1\mid t-1,i}.
$$

Here, \(\pi_{t\mid t-1,j}\) is the probability of state \(j\) before observing \(r_t\), and \(p_{ij}\) is the transition probability from state \(i\) to state \(j\). After \(r_t\) is observed, the filtered probability becomes

$$
\pi_{t\mid t,j}
=\frac{f_j(r_t)\pi_{t\mid t-1,j}}
{\sum_{\ell=0}^{1}f_\ell(r_t)\pi_{t\mid t-1,\ell}}.
$$

The function \(f_j(r_t)\) is the Gaussian density of the observed return under state \(j\). The denominator normalizes the two state probabilities so that they sum to one. The parameters are fitted only through December 2014 and are then held fixed. Filtering after that date uses each new return but does not re-estimate the means, variances, or transition probabilities.

In [4]:
full_model = MarkovRegression(returns["Daily log return (%)"].to_numpy(), k_regimes=2, trend="c", switching_variance=True)
filtered_raw = np.asarray(full_model.filter(parameters).filtered_marginal_probabilities)
smoothed_raw = np.asarray(full_model.smooth(parameters).smoothed_marginal_probabilities)

probabilities = pd.DataFrame({
    "Date": returns["Date"],
    "Daily log return (%)": returns["Daily log return (%)"],
    "Lower volatility probability": filtered_raw[:, low_state],
    "Higher volatility probability": filtered_raw[:, high_state],
    "Retrospective higher volatility probability": smoothed_raw[:, high_state],
})
probabilities["Training period"] = probabilities["Date"] <= training_cutoff
probabilities.to_csv(FIGURE_DATA / "state_probabilities.csv", index=False)

prefix_model = MarkovRegression(returns.loc[: len(training) - 1, "Daily log return (%)"].to_numpy(), k_regimes=2, trend="c", switching_variance=True)
prefix_filtered = np.asarray(prefix_model.filter(parameters).filtered_marginal_probabilities)
swapped_parameters = np.array([
    1 - parameters[1],
    1 - parameters[0],
    parameters[3],
    parameters[2],
    parameters[5],
    parameters[4],
])
swapped_filtered = np.asarray(full_model.filter(swapped_parameters).filtered_marginal_probabilities)

checks = pd.DataFrame([
    {"Check": "Filtered probabilities sum to one", "Maximum absolute error": np.max(np.abs(filtered_raw.sum(axis=1) - 1)), "Tolerance": 1e-12, "Result": np.max(np.abs(filtered_raw.sum(axis=1) - 1)) < 1e-12},
    {"Check": "Transition rows sum to one", "Maximum absolute error": np.max(np.abs(transition.sum(axis=1) - 1)), "Tolerance": 1e-12, "Result": np.max(np.abs(transition.sum(axis=1) - 1)) < 1e-12},
    {"Check": "Fitted variances are positive", "Maximum absolute error": max(0, -float(variances.min())), "Tolerance": 0, "Result": bool(np.all(variances > 0))},
    {"Check": "Later returns do not alter earlier filtered probabilities", "Maximum absolute error": np.max(np.abs(filtered_raw[: len(training)] - prefix_filtered)), "Tolerance": 1e-12, "Result": np.max(np.abs(filtered_raw[: len(training)] - prefix_filtered)) < 1e-12},
    {"Check": "Exchanging state labels only reverses probability columns", "Maximum absolute error": np.max(np.abs(filtered_raw - swapped_filtered[:, ::-1])), "Tolerance": 1e-12, "Result": np.max(np.abs(filtered_raw - swapped_filtered[:, ::-1])) < 1e-12},
])
save_table(checks, "markov_checks", "Numerical checks for the fitted state model and its filtered probabilities.", "tab:chapter14-markov-checks", fit=True)

extremes = probabilities.nlargest(10, "Higher volatility probability")[["Date", "Daily log return (%)", "Higher volatility probability"]].sort_values("Date")
save_table(extremes, "high_state_probability_dates", "Dates with the ten largest filtered probabilities of the higher volatility state.", "tab:chapter14-high-state-dates")

figure, axes = plt.subplots(2, 1, figsize=(11.2, 7.0), sharex=True, constrained_layout=True)
axes[0].plot(probabilities["Date"], probabilities["Daily log return (%)"], color=PALETTE["amber"], linewidth=0.58)
axes[0].axvline(training_cutoff, color=PALETTE["violet"], linestyle="--", linewidth=1.4, label="Fitting cutoff")
axes[0].set_ylabel("Return (%)")
axes[0].set_title("Returns and Filtered Probability of the Higher Volatility State")
axes[0].legend()
axes[1].plot(probabilities["Date"], probabilities["Higher volatility probability"], color=PALETTE["cyan"], linewidth=1.0)
axes[1].axvline(training_cutoff, color=PALETTE["violet"], linestyle="--", linewidth=1.4)
axes[1].set_ylabel("Filtered probability")
axes[1].set_xlabel("Date")
axes[1].set_ylim(-0.02, 1.02)
for axis in axes:
    finish_axis(axis)
save_figure(figure, "filtered_state_probability")
plt.close(figure)

contrast = probabilities.loc[probabilities["Date"].between("2006-01-01", "2010-12-31"), ["Date", "Higher volatility probability", "Retrospective higher volatility probability"]].copy()
contrast.to_csv(FIGURE_DATA / "filtered_smoothed_contrast.csv", index=False)
figure, axis = plt.subplots(figsize=(11.0, 5.5), constrained_layout=True)
axis.plot(contrast["Date"], contrast["Higher volatility probability"], color=PALETTE["cyan"], linewidth=1.35, label="Filtered, information through the date")
axis.plot(contrast["Date"], contrast["Retrospective higher volatility probability"], color=PALETTE["rose"], linewidth=1.25, alpha=0.9, label="Smoothed, later information allowed")
axis.set_xlabel("Date")
axis.set_ylabel("Higher volatility probability")
axis.set_ylim(-0.02, 1.02)
axis.set_title("Filtering and Retrospective Smoothing Answer Different Questions")
axis.legend()
finish_axis(axis)
save_figure(figure, "filtered_smoothed_contrast")
plt.close(figure)

,Check,Maximum absolute error,Tolerance,Result
0,Filtered probabilities sum to one,8.88178e-16,1e-12,Pass
1,Transition rows sum to one,0,1e-12,Pass
2,Fitted variances are positive,0,0,Pass
3,Later returns do not alter earlier filtered pr...,0,1e-12,Pass
4,Exchanging state labels only reverses probabil...,3.33067e-15,1e-12,Pass


,Date,Daily log return (%),Higher volatility probability
1577,2000-04-03,-7.942548,1
1767,2001-01-03,13.254645,1
1831,2001-04-05,8.545415,1
3711,2008-09-29,-9.58769,1
3723,2008-10-15,-8.850208,1
3755,2008-12-01,-9.380893,1
6594,2020-03-12,-9.909876,1
6595,2020-03-13,8.934695,1
6596,2020-03-16,-13.149154,1
7870,2025-04-09,11.478439,1


## Interpreting learned state labels

The numerical names zero and one have no economic meaning. This notebook calls the fitted state with the smaller conditional variance the lower volatility state and the other state the higher volatility state. A filtered probability uses returns available through the date being assessed. A smoothed probability can use later returns and is included only as a retrospective contrast.

The fitted states summarize two Gaussian components under constant transition probabilities. They do not prove that the market has exactly two economic regimes, identify a structural cause, or supply a trading rule. A high filtered probability means that the observed returns are more consistent with the fitted higher variance component under this model.

In [5]:
limits = pd.DataFrame([
    {"Item": "State label", "Permitted reading": "Ordered by fitted return variance", "Excluded reading": "A named economic regime"},
    {"Item": "Filtered probability", "Permitted reading": "Probability under the fitted model using information through the date", "Excluded reading": "A directly observed market state"},
    {"Item": "Smoothed probability", "Permitted reading": "Retrospective comparison using later observations", "Excluded reading": "An input to a live decision"},
    {"Item": "Transition probability", "Permitted reading": "Persistence under the constant transition model", "Excluded reading": "A permanent law of market behavior"},
    {"Item": "Public example", "Permitted reading": "Description of one index and period", "Excluded reading": "Evidence of forecasting or investment superiority"},
])
save_table(limits, "state_interpretation_limits", "Interpretation boundaries for the two state public illustration.", "tab:chapter14-state-interpretation-limits", fit=True)

,Item,Permitted reading,Excluded reading
0,State label,Ordered by fitted return variance,A named economic regime
1,Filtered probability,Probability under the fitted model using infor...,A directly observed market state
2,Smoothed probability,Retrospective comparison using later observations,An input to a live decision
3,Transition probability,Persistence under the constant transition model,A permanent law of market behavior
4,Public example,Description of one index and period,Evidence of forecasting or investment superiority


,Item,Permitted reading,Excluded reading
0,State label,Ordered by fitted return variance,A named economic regime
1,Filtered probability,Probability under the fitted model using infor...,A directly observed market state
2,Smoothed probability,Retrospective comparison using later observations,An input to a live decision
3,Transition probability,Persistence under the constant transition model,A permanent law of market behavior
4,Public example,Description of one index and period,Evidence of forecasting or investment superiority
